<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Tips & Techniques - Temporal Example
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style="font-size:24px;font-family:Arial;color:#00233C"><b>Introduction</b></p>

<p style="font-size:16px;font-family:Arial;color:#00233C">
The Teradata Enterprise Vector Store introduces powerful capabilities for managing and querying high-dimensional vector data directly within SQL. Among its emerging features, temporal vector embeddings offer a compelling approach to capturing time-aware semantics in unstructured data. While native support for temporal indexing is on the roadmap, current implementations rely on structured SQL workflows to capture temporal behavior. This notebook demonstrates how to construct and manage temporal vector tables using SQL functions such as TD_VectorNormalize, TD_VectorDistance, TD_Kmeans, and TD_HNSW, enabling semantic search and clustering over time-stamped data. These techniques are foundational for building intelligent, time-sensitive applications in domains like recommendation systems, anomaly detection, and document intelligence.
</p>

---

<p style="font-size:24px;font-family:Arial;color:#00233C"><b>Notebook Workflow Steps</b></p>

<div style="font-size:16px;font-family:Arial;color:#00233C">
<ol>
<li>
<b>Setup a base temporal table with commercial real estate data</b>
<ul>
<li>Connection to a Teradata Database environment</li>
<li>Create table commercial_real_estate_temporal to store the desired data</li>
</ul>
</li>

<li>
<b>Split table in two sections</b>
<ul>
<li>Create set A from commercial_real_estate_example</li>
<li>Create set B from commercial_real_estate_example</li>
</ul>
</li>

<li>
<b>Initial Data Load (SetA)</b>
<ul>
<li>Add SetA data to temporal table</li>
<li>Set the session valid-time period and count them</li>
</ul>
</li>

<li>
<b>Create the embeddings for SetA data</b>
<ul>
<li>Create temporary holding table</li>
<li>Create a table that holds the normalized embeddings</li>
</ul>
</li>

<li>
<b>Create the index for SetA data</b>
</li>

<li>
<b>Set up Kmeans for SetA Temporal table</b>
<ul>
<li>Build the model table using TD_Kmeans</li>
<li>Move the model data to the temporal table</li>
<li>Create a temporary table for kmeans centroids</li>
<li>Move the centroids into the temporal table</li>
</ul>
</li>

<li>
<b>Show a basic search using TD_VectorDistance</b>
<ul>
<li>Create a table for the question and embedding</li>
<li>Use TD_VectorDistance with the Kmeans clusters to ask the question of the data</li>
</ul>
</li>

<li>
<b>Add Setb data to the Base Table</b>
<ul>
<li>Update the data with SetB</li>
<li>Show the row counts with all the rows</li>
<li>Show the row count in the past</li>
</ul>
</li>

<li>
<b>Create embeddings for Set B and add to the base temporal table</b>
<ul>
<li>Add the data from Set B with the embeddings to the temporal table</li>
<li>Normalize the embeddings and add them to the temporal table</li>
</ul>
</li>

<li>
<b>Rebuild the index with SetA and SetB together</b>
</li>

<li>
<b>Show a basic search and row count</b>
<ul>
<li>Row counts</li>
<li>Build a Kmeans Model</li>
<li>Use TD_VectorDistance and the Kmeans Model table to run a Semantic Similarity Search</li>
</ul>
</li>


</ol>
</div>

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>1. Set up base temporal table </b>

<b style = 'font-size:18px;font-family:Arial;color:#00233C'>1. Connection to a Teradata database environment  </b>

In [ ]:
%connect vsdemo

In [ ]:
DATABASE <your_database_name>

Drop the table for real estate temporal in case it already exists

In [ ]:
DROP TABLE commercial_real_estate_temporal

<b style = 'font-size:18px;font-family:Arial;color:#00233C'>2. Create table `commercial_real_estate_temporal` to store the desired data  </b>

In [ ]:
CREATE MULTISET TABLE commercial_real_estate_temporal
    ,FALLBACK
    ,NO BEFORE JOURNAL
    ,NO AFTER JOURNAL
    ,CHECKSUM = DEFAULT
    ,DEFAULT MERGEBLOCKRATIO
    ,MAP = TD_MAP2
(
    id INTEGER,
    "Unnamed: 0" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    "title" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    nbn VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    address VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    text VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    area VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    "type" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    lattitude FLOAT,
    longitude FLOAT,
    Listing_Validity PERIOD(DATE) AS VALIDTIME
)
PRIMARY INDEX (id);

<div style="font-size:15px;font-family:Arial;color:#00233C;background:#fff8e6;padding:12px;border-radius:5px;border:1px solid #ffd700">
<b>Note:</b> The <code>Listing_Validity</code> column defines the date range during which each property listing is considered active or valid.
</div>

#### 💻 Syntax example

 ```sql
CREATE MULTISET TABLE commercial_real_estate_temporal
    ,FALLBACK
    ,NO BEFORE JOURNAL
    ,NO AFTER JOURNAL
    ,CHECKSUM = DEFAULT
    ,DEFAULT MERGEBLOCKRATIO
    ,MAP = TD_MAP2
(
    id INTEGER,
    "Unnamed: 0" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    "title" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    nbn VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    address VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    text VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    area VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    "type" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    lattitude FLOAT,
    longitude FLOAT,
    Listing_Validity PERIOD(DATE) AS VALIDTIME
)
PRIMARY INDEX (id);
```

⚠️ **ONLY** Execute if reloading data

In [ ]:
DELETE FROM commercial_real_estate_temporal;

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>2. Split Table Into Two Sections</b>

### 🗂️ Tables and Data Preparation

Break the data into two tables so that we can add data. **Capture** the timestamp as we have to know old / new times

1. **Create table Set A**
   - Create a table for set A from the commercial_real_estate_example table
   - View row count for Set A of data


❗Drop table for setA if it already exists

In [ ]:
DROP TABLE commercial_real_estate_setA

Create the table for Set A

In [ ]:
CREATE TABLE commercial_real_estate_setA AS (
    SELECT * 
    FROM commercial_real_estate_example
    WHERE HASHAMP(HASHBUCKET(HASHROW(id))) MOD 10 < 7 -- Assign 70 percent of rows to Set A
) WITH DATA;

Verify the number of rows in Set A 

In [ ]:
SELECT count(*) FROM commercial_real_estate_setA;

2. **Create a separate set of data SetB**
   - Assign 30 percent of rows in commercial_real_estate_example to set B
   - View row count for Set B of data

❗Drop table for setB if it already exists

In [ ]:
DROP TABLE commercial_real_estate_setB

In [ ]:
CREATE TABLE commercial_real_estate_setB AS (
    SELECT * 
    FROM commercial_real_estate_example
    WHERE HASHAMP(HASHBUCKET(HASHROW(id))) MOD 10 >= 7 -- Assign 30 percent of rows to Set B
) WITH DATA;

Verify the number of rows in Set B

In [ ]:
SELECT count(*) FROM commercial_real_estate_setB;

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>3. Initial Data Load (SetA) </b>

### 📝Add SetA data to temporal table
1. **Add first group of rows to the base temporal table using current time**

In [ ]:
NONSEQUENCED VALIDTIME
INSERT INTO commercial_real_estate_temporal 
    (id, "Unnamed: 0", "title", price, nbn, address, "text", area, "type", lattitude, longitude, Listing_Validity)
SELECT 
    id, "Unnamed: 0", "title", price, nbn, address, "text", area, "type", lattitude, longitude,
    PERIOD(DATE '2025-05-15', UNTIL_CHANGED)
FROM 
    commercial_real_estate_setA;

2. **Validate the row count added to the empty table.**
    - Set the session valid-time period
        - Limits all subsequent queries in the session to only see data that was valid between January 1, 2025 and May 16, 2025.
    - Count rows valid in that period
    - View sample 10 rows from the temporal table

In [ ]:
 set session validtime ( PERIOD(TIMESTAMP '2025-01-01 00:00:00.000', TIMESTAMP '2025-05-16 00:00:00.000'));

In [ ]:
SELECT count(*) FROM commercial_real_estate_temporal;

In [ ]:
select * from commercial_real_estate_temporal sample 10;

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>4. Create Embeddings for SetA data </b>

<div style="text-align: center;">
  <img src="VSData1b.png" alt="Alt Text" style="width:65%;">
</div>

1. **Create a temporary holding table** 
    - Here we create a table for the embeddings to insert into the final table
    - We need to create Embeddings into staging table

❗Drop table for temporal index temp if it already exists

In [ ]:
DROP TABLE vectorstore_commercial_real_estate_temporal_index_temp

Create a table to store the data and the embeddings. Use the AI_TextEmbeddings function to generate embeddings for the data.

In [ ]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index_temp AS (
SELECT id, price, Embedding as Vector_Index, Message 
FROM AI_TEXTEMBEDDINGS(   
    ON (
        SELECT 
            dt.id, 
            CONCAT(
                dt.text, ' ', 
                dt.address, ' ', 
                'Price: ', CAST(dt.price AS VARCHAR(50))
            ) AS text, 
            dt.price, 
            dt."title", 
            td_byone() 
        FROM commercial_real_estate_example dt 
        -- SAMPLE 1
    ) AS InputTable PARTITION BY TD_BYONE()
    USING 
        authorization(AWSEmbeddingsAuth)
        TextColumn('text')
        ApiType('aws')
        REGION('us-west-2')
        ModelName('amazon.titan-embed-text-v1')
        outputformat('vector')
        Accumulate(' "id" ', ' "title" ', ' "price" ')
) AS dt
) WITH DATA PRIMARY INDEX (id);

#### 💻 Syntax example

```sql
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index_temp ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
      Vector_Index SYSUDTLIB.Vector,
      Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC)
PRIMARY INDEX ( id );
```

**View** a sample of the data

In [ ]:
SELECT id, price, vector_index, vector_index as vector_index_normalized, Message 
            FROM vectorstore_commercial_real_estate_temporal_index_temp sample 1
            WHERE Vector_Index IS NOT NULL

❗Drop the table if it already exists

In [ ]:
DROP TABLE vectorstore_commercial_real_estate_temporal_index_temp_II

---

2. **Create a table and store the normalized embeddings in that table** 
    - Use TD_VectorNormalize to normalize the embeddings

In [ ]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index_temp_II AS (
SELECT id, price, vector_index, vector_index_normalized, Message 
FROM TD_Vectornormalize(
    ON (
           SELECT id, price, vector_index, vector_index as vector_index_normalized, Message 
            FROM vectorstore_commercial_real_estate_temporal_index_temp 
            WHERE vector_index IS NOT NULL
    ) AS InputTable
                USING
                IDColumns('id')
                TargetColumns('vector_index_normalized')
                Accumulate('price', 'vector_index', 'message')
                Approach('UNITVECTOR')
                EmbeddingSize(1536)
                ) AS dt
) WITH DATA PRIMARY INDEX (id);

#### 💻 Syntax example

```sql
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index_temp_II ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
      vector_index SYSUDTLIB.Vector,
      vector_index_normalized SYSUDTLIB.Vector,
      Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC)
PRIMARY INDEX ( id );
```

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>5. Create the index for SetA data </b>

#### 💻 Syntax Example

```sql
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index_temp ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
      Vector_Index SYSUDTLIB.Vector,
      Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC)
PRIMARY INDEX ( id );
```

❗Drop table for the temporal index if it already exists

In [ ]:
DROP TABLE vectorstore_commercial_real_estate_temporal_index

1. **Add the embeddings to the final temporal index table as a create table**

In [ ]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index (
      id,
      price,
      vector_index,
      vector_index_normalized,
      Message ,
      Listing_Validity 
      ) AS (
      NONSEQUENCED VALIDTIME PERIOD (DATE '2025-05-15', UNTIL_CHANGED)
      SELECT *
      FROM vectorstore_commercial_real_estate_temporal_index_temp_II)
   WITH DATA PRIMARY INDEX(id);

#### 💻 Syntax Example

```sql
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
      vector_index SYSUDTLIB.Vector,
      vector_index_normalized SYSUDTLIB.Vector,
      Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC,
      Listing_Validity PERIOD(DATE) AS VALIDTIME)
PRIMARY INDEX ( id );
```

---
**Congratulations!** The data has been stored in a temporal table and the embeddings index has been placed in a temporal index.


<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>6. Set up Kmeans for Set A temporal table </b>

#### Setup VS Kmeans Model


<div style="text-align: center;">
  <img src="VSDataModel2.png"alt="Alt Text" style="width:60%;">
</div>

#### Build a Kmeans model
1. **Build the model table using TD_Kmeans**
2. **Move the model data to the temporal table**
3. **Create a temporary table for kmeans centroids**
4. **Move the centroids into the temporal table**

❗Drop the table if it already exists

In [ ]:
DROP TABLE vectorstore_commercial_real_estate_kmeans_model_temp

1. **Use TD_Kmeans to create a Kmeans model with the index data**

In [ ]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_kmeans_model_temp as (
SELECT * FROM TD_KMEANS (
  ON vectorstore_commercial_real_estate_temporal_index AS InputTable
  USING
  IdColumn('id')
  TargetColumns('Vector_Index')
  InitialCentroidsMethod('RANDOM')
  NumClusters(10)
  Seed(0)
  StopThreshold(0.0395)
  MaxIterNum(10)
  NumInit(1)
  EmbeddingSize(1536)
) AS dt) WITH DATA NO PRIMARY INDEX

#### 💻 Syntax example

```sql
CREATE MULTISET TABLE vectorstore_commercial_real_estate_kmeans_model_temp ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      td_clusterid_kmeans BIGINT,
      vector_index SYSUDTLIB.Vector,
      td_size_kmeans BIGINT,
      td_withinss_kmeans FLOAT,
      id BYTEINT,
      td_modelinfo_kmeans VARCHAR(128) CHARACTER SET LATIN NOT CASESPECIFIC)
NO PRIMARY INDEX ;
```

❗Drop a table if it already exists

In [ ]:
DROP TABLE vectorstore_commercial_real_estate_temporal_model

2. **Move the model data to the temporal table**

In [ ]:
   CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_model (
      td_clusterid_kmeans,
      vector_index,
      td_size_kmeans,
      td_withinss_kmeans,
      id,
      td_modelinfo_kmeans,
      Listing_Validity 
      ) AS (
      NONSEQUENCED VALIDTIME PERIOD(DATE '2025-05-15', UNTIL_CHANGED)
      SELECT *
      FROM vectorstore_commercial_real_estate_kmeans_model_temp)
   WITH DATA
   PRIMARY INDEX(td_clusterid_kmeans);

#### 💻 Syntax example

```sql
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_model ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      td_clusterid_kmeans BIGINT,
      vector_index SYSUDTLIB.Vector,
      td_size_kmeans BIGINT,
      td_withinss_kmeans FLOAT,
      id BYTEINT,
      td_modelinfo_kmeans VARCHAR(128) CHARACTER SET LATIN NOT CASESPECIFIC,
      Listing_Validity PERIOD(DATE) AS VALIDTIME)
PRIMARY INDEX ( td_clusterid_kmeans );
```

❗Drop the table if it already exists

In [ ]:
DROP TABLE vectorstore_commercial_real_estate_temporal_centroids_temp

3. **Create a temporary table for kmeans centroids**

In [ ]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_centroids_temp AS (
SELECT id, Vector_Index, td_clusterid_kmeans as clusterID FROM TD_KMEANSPREDICT(
  ON vectorstore_commercial_real_estate_temporal_index AS InputTable
  ON vectorstore_commercial_real_estate_kmeans_model_temp AS ModelTable DIMENSION
  USING
  Accumulate( 'Vector_Index')
) AS dt) WITH DATA PRIMARY INDEX (clusterID);

#### 💻 Syntax example

```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_centroids_temp ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      vector_index SYSUDTLIB.Vector,
      clusterID BIGINT)
PRIMARY INDEX ( clusterID );
```

❗Drop the table if it already exists

In [ ]:
DROP TABLE vectorstore_commercial_real_estate_temporal_centroids

4. **Move the centroids into the temporal table**

In [ ]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_centroids (
      id,
      vector_index,
      clusterID,
      Listing_Validity 
      ) AS (
      NONSEQUENCED VALIDTIME PERIOD(DATE '2025-05-15', UNTIL_CHANGED)
      SELECT *
      FROM vectorstore_commercial_real_estate_temporal_centroids_temp)
   WITH DATA
   PRIMARY INDEX(id);

#### 💻 Syntax example

```sql
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_centroids ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      vector_index SYSUDTLIB.Vector,
      clusterID BIGINT)
PRIMARY INDEX ( clusterID );
```

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>7. Show a basic search using TD_VectorDistance</b>

1. **Create a table for the question and embedding**
2. **Use TD_VectorDistance with the Kmeans clusters to ask the question of the data**

#### 1. Create a table for the question and the corresponding embedding

❗Drop the question vector table if it exists

In [ ]:
DROP TABLE question_vector

In [ ]:
CREATE MULTISET TABLE question_vector  AS (
SELECT id, Embedding 
FROM AI_TEXTEMBEDDINGS(   
    ON (
        SELECT
            1 as ID,
            'I need properties on Harrington St.' as "Text"
    ) AS InputTable 
    USING 
        authorization(AWSEmbeddingsAuth)
        TextColumn('text')
        ApiType('aws')
        REGION('us-west-2')
        ModelName('amazon.titan-embed-text-v1')
        outputformat('vector')
        Accumulate(' "id" ')
) AS dt
) WITH DATA PRIMARY INDEX (id);

#### 💻 Syntax example

```sql
CREATE MULTISET TABLE question_vector ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      ID BYTEINT,
      Embedding SYSUDTLIB.Vector)
PRIMARY INDEX ( ID );
```

#### 2. Use TD_VectorDistance with the Kmeans clusters to ask the question of the data

In [ ]:
WITH ClusterIDs AS (
    -- Fetch clusterID values dynamically
    SELECT reference_id FROM TD_VECTORDISTANCE(
                ON question_vector AS TargetTable DIMENSION
                ON (
                    CURRENT VALIDTIME
                    SELECT td_clusterid_kmeans as clusterID, vector_index AS centroid 
                        FROM vectorstore_commercial_real_estate_temporal_model 
                        WHERE td_clusterid_kmeans IS NOT NULL) AS ReferenceTable
                USING
                TargetIDColumn('ID')
                TargetFeatureColumns('Embedding')
                RefIDColumn('clusterID')
                RefFeatureColumns('centroid')
                DistanceMeasure('EUCLIDEAN')
                OutputSimilarity('t')
                Topk(3)
                LargeReferenceInput('t')
                EmbeddingSize(1536)
                ) AS dt 
)
CURRENT VALIDTIME
SELECT t.id, t."title", t.price, vd.similarity
FROM commercial_real_estate_temporal t
JOIN (
    SELECT reference_id, similarity
    FROM TD_VECTORDISTANCE(
        ON question_vector AS TargetTable DIMENSION
        ON (
            CURRENT VALIDTIME
            SELECT * 
            FROM vectorstore_commercial_real_estate_temporal_centroids 
            WHERE clusterID IN (SELECT reference_id FROM ClusterIDs)
        ) AS ReferenceTable
        USING
            TargetIDColumn('id')
            TargetFeatureColumns('embedding')
            RefIDColumn('id')
            RefFeatureColumns('vector_index')
            DistanceMeasure('euclidean')
            OutputSimilarity('t')
            Topk(10)
            EmbeddingSize(1536)
            LargeReferenceInput('t')
    ) AS dt1
) vd
ON t.id = vd.reference_id
ORDER BY vd.similarity DESC;

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>8. Add SetB data to Base Table</b>
    
1. **Update data**
2. **Show the row counts with all the rows**
3. **Show the row count in the past**

#### 1. Update data with the data from Set B

In [ ]:
NONSEQUENCED VALIDTIME
INSERT INTO commercial_real_estate_temporal 
    (id, "Unnamed: 0", "title", price, nbn, address, "text", area, "type", lattitude, longitude, Listing_Validity)
SELECT 
    id, "Unnamed: 0", "title", price, nbn, address, "text", area, "type", lattitude, longitude,
    PERIOD(DATE '2025-06-01', DATE '9999-01-01')
FROM 
    commercial_real_estate_setB;

#### 2. Row count for all rows

In [ ]:
CURRENT VALIDTIME
SELECT Count(*) from commercial_real_estate_temporal;

#### 3. Row count in the past

In [ ]:
SEQUENCED VALIDTIME PERIOD(timestamp'2025-01-01 00:00:00.000000+00:00', timestamp'2025-05-30 00:00:00.000000+00:00')
SELECT Count(*) from commercial_real_estate_temporal;

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>9. Create embeddings for Set B and add to the base temporal table</b>

1. **Add the data from Set B with the embeddings to the temporal table**
2. **Normalize the embeddings and add them to the temporal table**

❗Drop the index temp table because you are reusing it

In [ ]:
DROP TABLE vectorstore_commercial_real_estate_temporal_index_temp

#### 1. Add Data from Set B with embeddings to the temporal index temp table

In [ ]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index_temp AS (
SELECT id, price, Embedding as Vector_Index, Message 
FROM AI_TEXTEMBEDDINGS(   
    ON (
        SELECT 
            dt.id, 
            CONCAT(
                dt.text, ' ', 
                dt.address, ' ', 
                'Price: ', CAST(dt.price AS VARCHAR(50))
            ) AS text, 
            dt.price, 
            dt."title", 
            td_byone() 
        FROM commercial_real_estate_setB dt 
        -- SAMPLE 1
    ) AS InputTable PARTITION BY TD_BYONE()
    USING 
        authorization(AWSEmbeddingsAuth)
        TextColumn('text')
        ApiType('aws')
        REGION('us-west-2')
        ModelName('amazon.titan-embed-text-v1')
        outputformat('vector')
        Accumulate(' "id" ', ' "title" ', ' "price" ')
) AS dt
) WITH DATA PRIMARY INDEX (id);

❗Drop the table for the index temp II because you are reusing it

In [ ]:
DROP TABLE vectorstore_commercial_real_estate_temporal_index_temp_II

#### 2. Add the normalized embeddings to the index temp II table

In [ ]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index_temp_II AS (
SELECT id, price, vector_index, vector_index_normalized, Message 
FROM TD_Vectornormalize(
    ON (
           SELECT id, price, vector_index, vector_index as vector_index_normalized, Message 
            FROM vectorstore_commercial_real_estate_temporal_index_temp 
            WHERE vector_index IS NOT NULL
    ) AS InputTable
                USING
                IDColumns('id')
                TargetColumns('vector_index_normalized')
                Accumulate('price', 'vector_index', 'message')
                Approach('UNITVECTOR')
                EmbeddingSize(1536)
                ) AS dt
) WITH DATA PRIMARY INDEX (id);

#### 💻 Syntax example

```sql
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
      vector_index SYSUDTLIB.Vector,
      vector_index_normalized SYSUDTLIB.Vector,
      Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC,
      Listing_Validity PERIOD(DATE) AS VALIDTIME)
PRIMARY INDEX ( id );
```

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>10. Rebuild the index with SetA and SetB together</b>

#### Move the embeddings into the temporal table correctly

In [ ]:
NONSEQUENCED VALIDTIME
INSERT INTO vectorstore_commercial_real_estate_temporal_index 
    (id, price, vector_index, vector_index_normalized, Message, Listing_Validity)
SELECT 
    id, price, vector_index, vector_index_normalized, Message,
    PERIOD(DATE '2025-06-01', DATE '9999-01-01')
FROM 
    vectorstore_commercial_real_estate_temporal_index_temp_II;

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>11. Show a basic search and row count</b>

#### Row counts
1. **View the row count of the temporal index table**
2. **Show the row count in the past**

#### Build a Kmeans model
1. **Build the model table using TD_Kmeans**
2. **Move the model data to the temporal table**
3. **Create a temporary table for kmeans centroids**
4. **Move the centroids into the temporal table**

#### Use TD_VectorDistance and the Kmeans model table to run a Semantic Similarity Search
1. **Run a Semantic Similarity Search**

<b style = 'font-size:20px;font-family:Arial;color:#00233C'>Row Counts</b>

#### 1. View the row count of the temporal index table

In [ ]:
CURRENT VALIDTIME
SELECT Count(*) from vectorstore_commercial_real_estate_temporal_index;

#### 2. Show the row count in the past

In [ ]:
SEQUENCED VALIDTIME PERIOD(date'2011-01-01', date'2025-05-16')
SELECT count(*) from vectorstore_commercial_real_estate_temporal_index;

<b style = 'font-size:20px;font-family:Arial;color:#00233C'>Build a Kmeans Model</b>

❗Drop the kmeans model temp table as you are reusing it

In [ ]:
DROP TABLE vectorstore_commercial_real_estate_kmeans_model_temp

#### 1. Build the Kmeans Model table using TD_Kmeans

In [ ]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_kmeans_model_temp as (
SELECT * FROM TD_KMEANS (
  ON vectorstore_commercial_real_estate_temporal_index AS InputTable
  USING
  IdColumn('id')
  TargetColumns('Vector_Index')
  InitialCentroidsMethod('RANDOM')
  NumClusters(10)
  Seed(0)
  StopThreshold(0.0395)
  MaxIterNum(10)
  NumInit(1)
  EmbeddingSize(1536)
) AS dt) WITH DATA NO PRIMARY INDEX

❗**Delete** model data as you are reusing the table name

In [ ]:
DELETE vectorstore_commercial_real_estate_temporal_model

#### 💻 Syntax example

```sql
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_model ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      td_clusterid_kmeans BIGINT,
      vector_index SYSUDTLIB.Vector,
      td_size_kmeans BIGINT,
      td_withinss_kmeans FLOAT,
      id BYTEINT,
      td_modelinfo_kmeans VARCHAR(128) CHARACTER SET LATIN NOT CASESPECIFIC,
      Listing_Validity PERIOD(DATE) AS VALIDTIME)
PRIMARY INDEX ( td_clusterid_kmeans );
```

#### 2. Move the model data into the temporal model table

In [ ]:
NONSEQUENCED VALIDTIME
INSERT INTO vectorstore_commercial_real_estate_temporal_model 
    (td_clusterid_kmeans, vector_index, td_size_kmeans, td_withinss_kmeans, id, td_modelinfo_kmeans, Listing_Validity)
SELECT 
    td_clusterid_kmeans, vector_index, td_size_kmeans, td_withinss_kmeans, id, td_modelinfo_kmeans,
    PERIOD(DATE '2025-06-01', DATE '9999-01-01')
FROM 
    vectorstore_commercial_real_estate_kmeans_model_temp;

❗Drop the temporal centroids table as you are reusing it

In [ ]:
DROP TABLE vectorstore_commercial_real_estate_temporal_centroids_temp

#### 3. Create a temporary table for the Kmeans centroids

In [ ]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_centroids_temp AS (
SELECT id, Vector_Index, td_clusterid_kmeans as clusterID FROM TD_KMEANSPREDICT(
  ON vectorstore_commercial_real_estate_temporal_index AS InputTable
  ON vectorstore_commercial_real_estate_kmeans_model_temp AS ModelTable DIMENSION
  USING
  Accumulate( 'Vector_Index')
) AS dt) WITH DATA PRIMARY INDEX (clusterID);

#### 💻 Syntax example

```sql
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_centroids ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      vector_index SYSUDTLIB.Vector,
      clusterID BIGINT,
      Listing_Validity PERIOD(DATE) AS VALIDTIME)
PRIMARY INDEX ( id );
```

#### 4. Move the centroids into the temporal table

In [ ]:
NONSEQUENCED VALIDTIME
INSERT INTO vectorstore_commercial_real_estate_temporal_centroids 
    (id, vector_index, clusterID, Listing_Validity)
SELECT 
    id, vector_index, clusterID,
    PERIOD(DATE '2025-06-01', DATE '9999-01-01')
FROM 
    vectorstore_commercial_real_estate_temporal_centroids_temp;

Set the session validtime

In [ ]:
 set session validtime ( PERIOD(TIMESTAMP '2025-01-01 00:00:00.000', TIMESTAMP '2025-05-01 00:00:00.000'));

<b style = 'font-size:20px;font-family:Arial;color:#00233C'>Vector Distance with Kmeansl</b>
<p style = 'font-size:16px;font-family:Arial;color:#00233C'>Ask the same question asked above based on the temporal data. The question is stored in the question_vector table.</p>

In [ ]:
WITH ClusterIDs AS (
    -- Fetch clusterID values dynamically
    SELECT reference_id FROM TD_VECTORDISTANCE(
                ON question_vector AS TargetTable DIMENSION
                ON (    
                        CURRENT VALIDTIME
                        SELECT td_clusterid_kmeans as clusterID, vector_index AS centroid 
                        FROM vectorstore_commercial_real_estate_temporal_model 
                        WHERE td_clusterid_kmeans IS NOT NULL) AS ReferenceTable
                USING
                TargetIDColumn('ID')
                TargetFeatureColumns('Embedding')
                RefIDColumn('clusterID')
                RefFeatureColumns('centroid')
                DistanceMeasure('EUCLIDEAN')
                OutputSimilarity('t')
                Topk(3)
                LargeReferenceInput('t')
                EmbeddingSize(1536)
                ) AS dt 
)
CURRENT VALIDTIME
SELECT t.id, t."title", t.price, vd.similarity
FROM commercial_real_estate_temporal t
JOIN (
    SELECT reference_id, similarity
    FROM TD_VECTORDISTANCE(
        ON question_vector AS TargetTable DIMENSION
        ON (
            CURRENT VALIDTIME
            SELECT * 
            FROM vectorstore_commercial_real_estate_temporal_centroids 
            WHERE clusterID IN (SELECT reference_id FROM ClusterIDs)
        ) AS ReferenceTable
        USING
            TargetIDColumn('id')
            TargetFeatureColumns('embedding')
            RefIDColumn('id')
            RefFeatureColumns('vector_index')
            DistanceMeasure('euclidean')
            OutputSimilarity('t')
            Topk(10)
            EmbeddingSize(1536)
            LargeReferenceInput('t')
    ) AS dt1
) vd
ON t.id = vd.reference_id
ORDER BY vd.similarity ASC;